# **Load** **Data**

In [1]:
!pip install gradio -q

In [2]:
import pandas as pd
import numpy as np
import re
import math
import nltk
import gradio as gr
from collections import Counter
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
nltk.download("stopwords")
from sklearn.metrics.pairwise import cosine_similarity


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
df=pd.read_csv("/content/movies_preprocessed.csv")
df["processed_text"]=df["processed_text"].fillna("")
df.head()

,doc_id,title,type,year,rating,original_abstract,processed_text
0,doc_1,7 Dogs,movie,2026.0,0.000,Interpol agent Khalid Al-Azzazi joins forces w...,7 dog interpol agent khalid al azzazi join for...
1,doc_2,Karmouz War,movie,2018.0,5.942,"Alexandria, Egypt, 1940. Three young Egyptians...",karmouz war alexandria egypt 1940 three young ...
2,doc_3,EgyBest,movie,2026.0,0.000,Two friends dive into the digital underworld c...,egybest two friend dive digit underworld chase...
3,doc_4,Love Story,movie,2019.0,7.500,"Youssef, whose brother is trying to push him t...",love stori youssef whose brother tri push marr...
4,doc_5,Incident of Dishonor,movie,1971.0,0.000,One of the beauties of the estate lives with h...,incid dishonor one beauti estat live sister br...


# **Build Inverted Index**

In [4]:
documents={}
for i, row in df.iterrows():
    documents[row["doc_id"]]=row["processed_text"]
inverted_index={}
for doc_id, text in documents.items():
    words=text.split()
    word_freq=Counter(words)
    for word, freq in word_freq.items():
        if word not in inverted_index:
            inverted_index[word]=[]
        inverted_index[word].append((doc_id, freq))
print("Total documents:", len(documents))
print("Total unique terms:", len(inverted_index))

Total documents: 10042
Total unique terms: 22255


# **Query Processing**

In [5]:
stemmer=SnowballStemmer("english")
stop_words=set(stopwords.words("english"))
def preprocess_query(query):
    query=query.lower()
    query=re.sub(r"[^a-zA-Z0-9\s]", " ", query)
    tokens=query.split()
    tokens=[w for w in tokens if w not in stop_words]
    tokens=[stemmer.stem(w) for w in tokens]
    return tokens

In [6]:
def retrieve_documents(query_terms, inverted_index):
    matched_docs=None
    for term in query_terms:
        if term not in inverted_index:
            return []
        docs_for_term=set()
        for doc_id, freq in inverted_index[term]:
            docs_for_term.add(doc_id)
        if matched_docs is None:
            matched_docs=docs_for_term
        else:
            matched_docs=matched_docs.intersection(docs_for_term)
    return list(matched_docs)

In [7]:
def calculate_tfidf(doc_id, query_terms, inverted_index, total_docs):
    score=0
    for term in query_terms:
        postings=inverted_index.get(term, [])
        df_term=len(postings)
        if df_term==0:
            continue
        idf=math.log(total_docs / df_term)
        tf=0
        for d_id, freq in postings:
            if d_id==doc_id:
                tf=freq
                break
        score += tf * idf
    return score

In [8]:
def rank_documents(query, inverted_index, total_docs):
    query_terms=preprocess_query(query)
    docs=retrieve_documents(query_terms, inverted_index)
    results=[]
    for doc_id in docs:
        score=calculate_tfidf(doc_id, query_terms, inverted_index, total_docs)
        results.append((doc_id, score))
    results=sorted(results, key=lambda x: x[1], reverse=True)
    return results

# **Test**

In [9]:
query="crime drug"
results=rank_documents(query, inverted_index, len(df))
print("Query:", query)
print("Number of results:", len(results))
print("\nTop Results:")
for doc_id, score in results[:10]:
    row=df[df["doc_id"]==doc_id]
    if len(row) > 0:
        print("Doc ID:", doc_id)
        print("Title:", row.iloc[0]["title"])
        print("Score:", round(score, 4))
        print("-" * 40)

Query: crime drug
Number of results: 15

Top Results:
Doc ID: doc_2045
Title: The Emperor
Score: 21.0928
----------------------------------------
Doc ID: doc_3718
Title: The Killer Of Karmoz
Score: 10.5464
----------------------------------------
Doc ID: doc_2105
Title: ابن تحية عزوز
Score: 10.5464
----------------------------------------
Doc ID: doc_1
Title: 7 Dogs
Score: 7.0812
----------------------------------------
Doc ID: doc_8541
Title: The Death and Life of Bobby Z
Score: 7.0812
----------------------------------------
Doc ID: doc_9804
Title: Straight Out of Brooklyn
Score: 7.0812
----------------------------------------
Doc ID: doc_9263
Title: Jesus' Son
Score: 7.0812
----------------------------------------
Doc ID: doc_3471
Title: Executioner's sword
Score: 7.0812
----------------------------------------
Doc ID: doc_8161
Title: The Fighter
Score: 7.0812
----------------------------------------
Doc ID: doc_2709
Title: Wi Daa El Tareeq
Score: 7.0812
----------------------------

# **Query Expansion**

**BM25 Pseudo Relevance Feedback (PRF) — Infrastructure**

In [10]:
BM25_K1 = 1.5
BM25_B   = 0.75
doc_lengths = {
    doc_id: len(text.split())
    for doc_id, text in documents.items()
}
avg_doc_length = sum(doc_lengths.values()) / len(doc_lengths)
def compute_bm25_score(doc_id, query_terms, inverted_index, total_docs,
                       k1=BM25_K1, b=BM25_B):
    score   = 0.0
    doc_len = doc_lengths.get(doc_id, 0)
    for term in query_terms:
        postings = inverted_index.get(term, [])
        df_term  = len(postings)
        if df_term == 0:
            continue
        idf = math.log((total_docs - df_term + 0.5) / (df_term + 0.5) + 1)
        tf = next((freq for d_id, freq in postings if d_id == doc_id), 0)
        tf_norm = (tf * (k1 + 1)) / (
            tf + k1 * (1 - b + b * doc_len / avg_doc_length)
        )
        score += idf * tf_norm
    return score
print(f"BM25 infrastructure ready.")
print(f"  Corpus size        : {len(doc_lengths):,} documents")
print(f"  Average doc length : {avg_doc_length:.1f} tokens")
print(f"  BM25 k1={BM25_K1}, b={BM25_B}")


BM25 infrastructure ready.
  Corpus size        : 10,042 documents
  Average doc length : 30.4 tokens
  BM25 k1=1.5, b=0.75


**BM25-Based Pseudo Relevance Feedback (PRF)**

In [11]:
def pseudo_relevance_feedback_bm25(query_terms, initial_results,
                                   inverted_index, total_docs,
                                   top_k_docs=5, top_n_terms=10):
    if not initial_results:
        return []
    pseudo_relevant_docs = [doc_id for doc_id, _ in initial_results[:top_k_docs]]
    candidate_freq = Counter()
    for doc_id in pseudo_relevant_docs:
        text = documents.get(doc_id, "")
        candidate_freq.update(text.split())
    term_scores = {}
    for term, _ in candidate_freq.most_common(200):
        if term in query_terms or len(term) <= 2:
            continue
        aggregated_score = sum(
            compute_bm25_score(doc_id, [term], inverted_index, total_docs)
            for doc_id in pseudo_relevant_docs
        )
        term_scores[term] = aggregated_score
    sorted_terms = sorted(term_scores.items(), key=lambda x: x[1], reverse=True)
    expansion_terms = [
        term for term, score in sorted_terms[:top_n_terms]
        if score > 0
    ]
    return expansion_terms

In [13]:
def retrieve_documents_any(query_terms, inverted_index):
    matched_docs=set()
    for term in query_terms:
        for doc_id, freq in inverted_index.get(term, []):
            matched_docs.add(doc_id)
    return list(matched_docs)

# **Term Representation Extension**




In [14]:
def build_term_document_matrix(documents, max_terms=1000):
    corpus_counter=Counter()
    for text in documents.values():
        corpus_counter.update(text.split())
    vocabulary=[]
    for term, freq in corpus_counter.most_common(max_terms):
        if len(term) > 2:
            vocabulary.append(term)
    matrix=pd.DataFrame(0, index=vocabulary, columns=list(documents.keys()))
    for doc_id, text in documents.items():
        counts=Counter(text.split())
        for term in vocabulary:
            if term in counts:
                matrix.loc[term, doc_id]=counts[term]
    return matrix
term_doc_matrix=build_term_document_matrix(documents, max_terms=1000)
print("Term-document matrix shape:", term_doc_matrix.shape)
term_doc_matrix.head()

Term-document matrix shape: (986, 10042)


,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7,doc_8,doc_9,doc_10,...,doc_10033,doc_10034,doc_10035,doc_10036,doc_10037,doc_10038,doc_10039,doc_10040,doc_10041,doc_10042
love,0,0,1,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
life,0,0,0,0,0,0,0,0,2,1,...,0,0,1,0,0,0,0,0,0,0
man,0,1,0,0,1,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,0
get,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
one,0,2,1,1,1,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,1


In [15]:
def get_representation_terms(query_terms, term_doc_matrix, top_k=3):
    related_terms=[]
    for term in query_terms:
        if term in term_doc_matrix.index:
            term_vector=term_doc_matrix.loc[term]
            scores={}
            for other_term in term_doc_matrix.index:
                if other_term != term:
                    other_vector=term_doc_matrix.loc[other_term]
                    common_score=(term_vector * other_vector).sum()
                    if common_score > 0:
                        scores[other_term]=common_score
            sorted_terms=sorted(scores.items(), key=lambda x: x[1], reverse=True)
            for word, score in sorted_terms[:top_k]:
                if word not in related_terms:
                    related_terms.append(word)
    return related_terms
sample_terms=preprocess_query("crime drug")
print("Original terms:", sample_terms)
print("Related representation terms:", get_representation_terms(sample_terms, term_doc_matrix, top_k=3))

Original terms: ['crime', 'drug']
Related representation terms: ['murder', 'one', 'get', 'dealer', 'polic']


**Rank documents with expansion**

In [16]:
def rank_documents_with_expansion(query, inverted_index, total_docs, df,
                                  top_k_docs=5, top_n_terms=10):
    original_terms = preprocess_query(query)
    if not original_terms:
        return [], []
    initial_doc_ids = retrieve_documents_any(original_terms, inverted_index)
    if not initial_doc_ids:
        return [], original_terms
    initial_results = [
        (doc_id, compute_bm25_score(doc_id, original_terms,
                                    inverted_index, total_docs))
        for doc_id in initial_doc_ids
    ]
    initial_results.sort(key=lambda x: x[1], reverse=True)
    expansion_terms = pseudo_relevance_feedback_bm25(
        query_terms     = original_terms,
        initial_results = initial_results,
        inverted_index  = inverted_index,
        total_docs      = total_docs,
        top_k_docs      = top_k_docs,
        top_n_terms     = top_n_terms
    )
    expanded_query_terms = original_terms.copy()
    for term in expansion_terms:
        if term not in expanded_query_terms:
            expanded_query_terms.append(term)
    final_doc_ids = retrieve_documents_any(expanded_query_terms, inverted_index)
    if not final_doc_ids:
        final_doc_ids        = initial_doc_ids
        expanded_query_terms = original_terms
    results = [
        (doc_id, compute_bm25_score(doc_id, expanded_query_terms,
                                    inverted_index, total_docs))
        for doc_id in final_doc_ids
    ]
    results.sort(key=lambda x: x[1], reverse=True)
    return results, expanded_query_terms

**Test Query Expansion**

In [17]:
query="crime drug"
expanded_results, expanded_terms=rank_documents_with_expansion(
    query,
    inverted_index,
    len(df),
    df
)
print("Original Query:", query)
print("Expanded Query Terms:", expanded_terms)
print("Number of results:", len(expanded_results))
print("\nTop Results After Query Expansion:")
for doc_id, score in expanded_results[:10]:
    row=df[df["doc_id"]==doc_id]
    if len(row) > 0:
        print("Doc ID:", doc_id)
        print("Title:", row.iloc[0]["title"])
        print("Score:", round(score, 4))
        print("-" * 40)

Original Query: crime drug
Expanded Query Terms: ['crime', 'drug', 'dealer', 'tryin', 'azzazi', 'tier', 'compass', 'addict', 'ghali', 'dawood', 'coke', 'interpol']
Number of results: 650

Top Results After Query Expansion:
Doc ID: doc_1
Title: 7 Dogs
Score: 48.3768
----------------------------------------
Doc ID: doc_8971
Title: Easy Money
Score: 34.8595
----------------------------------------
Doc ID: doc_9263
Title: Jesus' Son
Score: 26.3186
----------------------------------------
Doc ID: doc_6478
Title: Get Rich or Die Tryin'
Score: 26.0411
----------------------------------------
Doc ID: doc_9888
Title: On the Outs
Score: 16.2276
----------------------------------------
Doc ID: doc_2045
Title: The Emperor
Score: 15.5737
----------------------------------------
Doc ID: doc_2529
Title: No return
Score: 15.1125
----------------------------------------
Doc ID: doc_9493
Title: Fetching Cody
Score: 14.6104
----------------------------------------
Doc ID: doc_8880
Title: Light Sleeper
Sc

# **User** **Interface**

**Search Function**

In [18]:
def search_engine(query, top_n=10):
    results, expanded_terms = rank_documents_with_expansion(
        query,
        inverted_index,
        len(df),
        df
    )
    print("Original Query:", query)
    print("Expanded Terms:", expanded_terms)
    print("-" * 50)
    if len(results)==0:
        print("No results found.")
        return
    print("Top Search Results:\n")
    for rank, (doc_id, score) in enumerate(results[:top_n], start=1):
        row=df[df["doc_id"] == doc_id]
        if len(row) > 0:
            title = row.iloc[0]["title"]
            print("Rank:", rank)
            print("Doc ID:", doc_id)
            print("Title:", title)
            if "type" in df.columns:
                print("Type:", row.iloc[0]["type"])
            if "year" in df.columns:
                print("Year:", row.iloc[0]["year"])
            print("Score:", round(score, 4))
            print("-" * 50)

**Basic User Input Interface**

In [19]:
user_query=input("Enter your search query: ")
search_engine(user_query, top_n=10)

Enter your search query: love story
Original Query: love story
Expanded Terms: ['love', 'stori', 'roxi', 'remad', 'happier', 'fabl', '1971', 'swap', 'capit', 'default', 'fisherman', 'drunk']
--------------------------------------------------
Top Search Results:

Rank: 1
Doc ID: doc_1266
Title: Love Story
Type: movie
Year: 1959.0
Score: 45.078
--------------------------------------------------
Rank: 2
Doc ID: doc_7436
Title: Capitalism: A Love Story
Type: movie
Year: 2009.0
Score: 23.8605
--------------------------------------------------
Rank: 3
Doc ID: doc_1816
Title: Love and Tears
Type: movie
Year: 1955.0
Score: 23.5134
--------------------------------------------------
Rank: 4
Doc ID: doc_3583
Title: Roxi
Type: movie
Year: 2024.0
Score: 18.7617
--------------------------------------------------
Rank: 5
Doc ID: doc_8893
Title: Lovely, Still
Type: movie
Year: 2008.0
Score: 15.514
--------------------------------------------------
Rank: 6
Doc ID: doc_1454
Title: The Chicest Man in Rox

**Test with Ready Queries**

In [20]:
test_queries=[
    "crime drug",
    "love story",
    "space adventure",
    "war soldier",
    "family drama"
]
for q in test_queries:
    print("=" * 70)
    search_engine(q, top_n=5)

Original Query: crime drug
Expanded Terms: ['crime', 'drug', 'dealer', 'tryin', 'azzazi', 'tier', 'compass', 'addict', 'ghali', 'dawood', 'coke', 'interpol']
--------------------------------------------------
Top Search Results:

Rank: 1
Doc ID: doc_1
Title: 7 Dogs
Type: movie
Year: 2026.0
Score: 48.3768
--------------------------------------------------
Rank: 2
Doc ID: doc_8971
Title: Easy Money
Type: movie
Year: 2010.0
Score: 34.8595
--------------------------------------------------
Rank: 3
Doc ID: doc_9263
Title: Jesus' Son
Type: movie
Year: 1999.0
Score: 26.3186
--------------------------------------------------
Rank: 4
Doc ID: doc_6478
Title: Get Rich or Die Tryin'
Type: movie
Year: 2005.0
Score: 26.0411
--------------------------------------------------
Rank: 5
Doc ID: doc_9888
Title: On the Outs
Type: movie
Year: 2004.0
Score: 16.2276
--------------------------------------------------
Original Query: love story
Expanded Terms: ['love', 'stori', 'roxi', 'remad', 'happier', 'fabl

# **Evaluation**

**Evaluation Function**

In [21]:
import time
def evaluate_search_engine(test_queries):
    evaluation_results=[]
    for query in test_queries:
        start_time=time.time()
        results, expanded_terms=rank_documents_with_expansion(
            query,
            inverted_index,
            len(df),
            df
        )
        end_time=time.time()
        search_time=end_time - start_time
        if len(results) > 0:
            top_doc_id=results[0][0]
            row = df[df["doc_id"]==top_doc_id]
            if len(row) > 0:
                top_title=row.iloc[0]["title"]
            else:
                top_title="Unknown"
        else:
            top_title="No result"
        evaluation_results.append({
            "Query": query,
            "Expanded Terms": ", ".join(expanded_terms),
            "Number of Results": len(results),
            "Search Time": round(search_time, 4),
            "Top Result": top_title
        })
    return pd.DataFrame(evaluation_results)

**Test Various Queries**

In [22]:
test_queries = [
    "crime drug",
    "love story",
    "space adventure",
    "war soldier",
    "family drama",
    "magic school",
    "robot future",
    "money business"
]
evaluation_df=evaluate_search_engine(test_queries)
evaluation_df

,Query,Expanded Terms,Number of Results,Search Time,Top Result
0,crime drug,"crime, drug, dealer, tryin, azzazi, tier, comp...",650,0.0745,7 Dogs
1,love story,"love, stori, roxi, remad, happier, fabl, 1971,...",2614,1.0279,Love Story
2,space adventure,"space, adventur, conquer, interstellar, harloc...",480,0.1217,Space Pirate Captain Harlock
3,war soldier,"war, soldier, battl, american, chromit, incheo...",876,0.2869,Operation Chromite
4,family drama,"famili, drama, social, aleik, wahba, besieg, a...",1913,1.1552,Heir Apparent
5,magic school,"magic, school, neverend, angelov, fantasia, gi...",524,0.0815,Practical Magic
6,robot future,"robot, futur, human, automata, jacq, vaucan, r...",641,0.0836,Automata
7,money business,"money, busi, fatiha, nahid, bag, boxer, fledg,...",761,0.1741,Had Fasel: Al Bahth An Fteeha


**Display Top Titles for Accuracy Check**

In [23]:
for query in test_queries:
    results, expanded_terms=rank_documents_with_expansion(
        query,
        inverted_index,
        len(df),
        df
    )
    print("=" * 70)
    print("Query:", query)
    print("Expanded Terms:", expanded_terms)
    print("Top 5 Titles:")
    if len(results)==0:
        print("No results found.")
    for doc_id, score in results[:5]:
        row=df[df["doc_id"] == doc_id]
        if len(row) > 0:
            print("-", row.iloc[0]["title"], "| Score:", round(score, 4))
    print("-" * 60)

Query: crime drug
Expanded Terms: ['crime', 'drug', 'dealer', 'tryin', 'azzazi', 'tier', 'compass', 'addict', 'ghali', 'dawood', 'coke', 'interpol']
Top 5 Titles:
- 7 Dogs | Score: 48.3768
- Easy Money | Score: 34.8595
- Jesus' Son | Score: 26.3186
- Get Rich or Die Tryin' | Score: 26.0411
- On the Outs | Score: 16.2276
------------------------------------------------------------
Query: love story
Expanded Terms: ['love', 'stori', 'roxi', 'remad', 'happier', 'fabl', '1971', 'swap', 'capit', 'default', 'fisherman', 'drunk']
Top 5 Titles:
- Love Story | Score: 45.078
- Capitalism: A Love Story | Score: 23.8605
- Love and Tears | Score: 23.5134
- Roxi | Score: 18.7617
- Lovely, Still | Score: 15.514
------------------------------------------------------------
Query: space adventure
Expanded Terms: ['space', 'adventur', 'conquer', 'interstellar', 'harlock', 'planet', 'zathura', 'robinson', 'ham', 'wormhol', 'pirat', 'earth']
Top 5 Titles:
- Space Pirate Captain Harlock | Score: 52.8992
- I

**Relevance Keywords**

In [24]:
relevance_keywords={
    "crime drug": ["crime", "criminal", "drug", "drugs", "murder", "police", "detective", "trafficking"],
    "love story": ["love", "romance", "romantic", "relationship", "marriage"],
    "space adventure": ["space", "alien", "planet", "galaxy", "universe", "adventure"],
    "war soldier": ["war", "battle", "soldier", "army", "military"],
    "family drama": ["family", "father", "mother", "children", "drama"],
    "magic school": ["magic", "wizard", "witch", "spell", "school", "student"],
    "robot future": ["robot", "machine", "android", "technology", "future"],
    "money business": ["money", "business", "rich", "bank", "company"]
}

**Relevance Function**

In [25]:
def is_relevant(doc_id, query, df):
    row=df[df["doc_id"]==doc_id]
    if len(row)==0:
        return False
    title=str(row.iloc[0]["title"]).lower()
    if "original_abstract" in df.columns:
        abstract=str(row.iloc[0]["original_abstract"]).lower()
    elif "abstract" in df.columns:
        abstract=str(row.iloc[0]["abstract"]).lower()
    else:
        abstract=""
    text=title + " " + abstract
    keywords=relevance_keywords.get(query, [])
    for keyword in keywords:
        if keyword in text:
            return True
    return False

**Precision@5 Function**

In [26]:
def precision_at_5(results, query, df):
    top_5=results[:5]
    if len(top_5)==0:
        return 0
    relevant_count=0
    for doc_id, score in top_5:
        if is_relevant(doc_id, query, df):
            relevant_count += 1
    return relevant_count / 5

**Add Precision@5 to Evaluation Table**

In [27]:
precision_scores=[]
for query in test_queries:
    results, expanded_terms = rank_documents_with_expansion(
        query,
        inverted_index,
        len(df),
        df
    )
    p5=precision_at_5(results, query, df)
    precision_scores.append(p5)
evaluation_df["Precision@5"] = precision_scores
evaluation_df

,Query,Expanded Terms,Number of Results,Search Time,Top Result,Precision@5
0,crime drug,"crime, drug, dealer, tryin, azzazi, tier, comp...",650,0.0745,7 Dogs,1.0
1,love story,"love, stori, roxi, remad, happier, fabl, 1971,...",2614,1.0279,Love Story,1.0
2,space adventure,"space, adventur, conquer, interstellar, harloc...",480,0.1217,Space Pirate Captain Harlock,1.0
3,war soldier,"war, soldier, battl, american, chromit, incheo...",876,0.2869,Operation Chromite,1.0
4,family drama,"famili, drama, social, aleik, wahba, besieg, a...",1913,1.1552,Heir Apparent,1.0
5,magic school,"magic, school, neverend, angelov, fantasia, gi...",524,0.0815,Practical Magic,1.0
6,robot future,"robot, futur, human, automata, jacq, vaucan, r...",641,0.0836,Automata,1.0
7,money business,"money, busi, fatiha, nahid, bag, boxer, fledg,...",761,0.1741,Had Fasel: Al Bahth An Fteeha,1.0


**Mean Precision**

In [28]:
mean_precision=evaluation_df["Precision@5"].mean()
print("Mean Precision@5:", round(mean_precision, 3))

Mean Precision@5: 1.0


# **Evaluation Metrics**

In [29]:
def count_relevant_documents(query, df):
    count=0
    for doc_id in df["doc_id"]:
        if is_relevant(doc_id, query, df):
            count += 1
    return count
def recall_at_5(results, query, df):
    total_relevant=count_relevant_documents(query, df)
    if total_relevant==0:
        return 0
    relevant_in_top_5=0
    for doc_id, score in results[:5]:
        if is_relevant(doc_id, query, df):
            relevant_in_top_5 += 1
    return relevant_in_top_5 / total_relevant
def f1_at_5(precision, recall):
    if precision + recall == 0:
        return 0
    return 2 * precision * recall / (precision + recall)
precision_scores=[]
recall_scores=[]
f1_scores=[]
for query in test_queries:
    results, expanded_terms=rank_documents_with_expansion(
        query,
        inverted_index,
        len(df),
        df
    )
    p5=precision_at_5(results, query, df)
    r5=recall_at_5(results, query, df)
    f5=f1_at_5(p5, r5)
    precision_scores.append(round(p5, 3))
    recall_scores.append(round(r5, 3))
    f1_scores.append(round(f5, 3))
evaluation_df["Precision@5"]=precision_scores
evaluation_df["Recall@5"]=recall_scores
evaluation_df["F1@5"]=f1_scores
evaluation_df

,Query,Expanded Terms,Number of Results,Search Time,Top Result,Precision@5,Recall@5,F1@5
0,crime drug,"crime, drug, dealer, tryin, azzazi, tier, comp...",650,0.0745,7 Dogs,1.0,0.004,0.007
1,love story,"love, stori, roxi, remad, happier, fabl, 1971,...",2614,1.0279,Love Story,1.0,0.002,0.004
2,space adventure,"space, adventur, conquer, interstellar, harloc...",480,0.1217,Space Pirate Captain Harlock,1.0,0.011,0.022
3,war soldier,"war, soldier, battl, american, chromit, incheo...",876,0.2869,Operation Chromite,1.0,0.004,0.008
4,family drama,"famili, drama, social, aleik, wahba, besieg, a...",1913,1.1552,Heir Apparent,1.0,0.002,0.004
5,magic school,"magic, school, neverend, angelov, fantasia, gi...",524,0.0815,Practical Magic,1.0,0.007,0.013
6,robot future,"robot, futur, human, automata, jacq, vaucan, r...",641,0.0836,Automata,1.0,0.020,0.040
7,money business,"money, busi, fatiha, nahid, bag, boxer, fledg,...",761,0.1741,Had Fasel: Al Bahth An Fteeha,1.0,0.004,0.007


# **Final User Interface**

In [30]:
def format_search_results(query, top_n):
    results, expanded_terms=rank_documents_with_expansion(
        query,
        inverted_index,
        len(df),
        df
    )
    summary=f"""
    <div class='summary-box'>
        <h3>Search Summary</h3>
        <p><b>Original Query:</b> {query}</p>
        <p><b>Expanded Terms:</b> {', '.join(expanded_terms)}</p>
        <p><b>Total Retrieved Documents:</b> {len(results)}</p>
    </div>
    """
    rows=[]
    for rank, (doc_id, score) in enumerate(results[:top_n], start=1):
        row=df[df["doc_id"]==doc_id]
        if len(row) > 0:
            title=row.iloc[0]["title"] if "title" in df.columns else "Unknown"
            movie_type=row.iloc[0]["type"] if "type" in df.columns else "-"
            year=row.iloc[0]["year"] if "year" in df.columns else "-"
            if "original_abstract" in df.columns:
                description=str(row.iloc[0]["original_abstract"])
            elif "abstract" in df.columns:
                description=str(row.iloc[0]["abstract"])
            else:
                description=str(row.iloc[0]["processed_text"])
            if len(description) > 260:
                description=description[:260] + "..."
            rows.append({
                "Rank": rank,
                "Doc ID": doc_id,
                "Title": title,
                "Type": movie_type,
                "Year": year,
                "TF-IDF Score": round(score, 4),
                "Description": description
            })
    if len(rows)==0:
        rows.append({
            "Rank": "-",
            "Doc ID": "-",
            "Title": "No results found",
            "Type": "-",
            "Year": "-",
            "TF-IDF Score": "-",
            "Description": "Try another query."
        })
    result_df=pd.DataFrame(rows)
    return summary, result_df
custom_css="""
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
}
#main-title {
    text-align: center;
    padding: 18px;
    border-radius: 18px;
    background: linear-gradient(135deg, #eef2ff, #fdf2f8);
    margin-bottom: 15px;
}
#search-box textarea {
    font-size: 18px !important;
    min-height: 75px !important;
}
#results-table {
    min-height: 520px !important;
}
.summary-box {
    padding: 18px;
    border-radius: 16px;
    background: #f8fafc;
    border: 1px solid #e5e7eb;
    font-size: 16px;
}
.footer-note {
    font-size: 14px;
    color: #475569;
    text-align: center;
}
"""
with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <div id='main-title'>
        <h1>Movie Search Engine</h1>
        <p>TF-IDF Ranking with Query Expansion and Term Representation</p>
    </div>
    """)
    with gr.Row():
        with gr.Column(scale=3):
            query_input=gr.Textbox(
                label="Enter Search Query",
                placeholder="Example: crime drug, love story, space adventure",
                elem_id="search-box"
            )
        with gr.Column(scale=1):
            top_n_input=gr.Slider(
                minimum=5,
                maximum=20,
                value=10,
                step=1,
                label="Number of Results"
            )
            search_button=gr.Button("Search", variant="primary", size="lg")
    summary_output=gr.HTML(label="Search Summary")
    results_output=gr.Dataframe(
        label="Ranked Search Results",
        elem_id="results-table",
        wrap=True,
        interactive=False
    )
    gr.Examples(
        examples=[
            ["crime drug", 10],
            ["love story", 10],
            ["space adventure", 10],
            ["war soldier", 10],
            ["magic school", 10]
        ],
        inputs=[query_input, top_n_input]
    )
    gr.HTML("<p class='footer-note'>The query is preprocessed and expanded using BM25-based Pseudo Relevance Feedback (PRF). Expansion terms are extracted automatically from top-ranked pseudo-relevant documents and weighted by BM25 scores. Final ranking uses BM25.</p>")
    search_button.click(
        fn=format_search_results,
        inputs=[query_input, top_n_input],
        outputs=[summary_output, results_output]
    )
    query_input.submit(
        fn=format_search_results,
        inputs=[query_input, top_n_input],
        outputs=[summary_output, results_output]
    )
demo.launch(share=True)

/tmp/ipykernel_4840/2396627264.py:84: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_4840/2396627264.py:84: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9171c070e795c2b01d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
